In [24]:
import os
import numpy as np
import pandas as pd
import joblib

# Regression Models
from sklearn.linear_model import Ridge
# Evaluation Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [25]:
# Setup path
BASE_PATH = 'dataset'
OUTPUT_FOLDER = os.path.join(BASE_PATH, 'output_pipeline')
ARTIFACTS_FOLDER = os.path.join(BASE_PATH, 'artifacts')

In [26]:
# Load fitur (X)
X_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_train.parquet'))
X_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_val.parquet'))
X_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_test.parquet'))

In [27]:
# Load target (y) - convert ke Series untuk sklearn
y_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_train.parquet')).iloc[:, 0]
y_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_val.parquet')).iloc[:, 0]
y_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_test.parquet')).iloc[:, 0]

In [28]:
# Load metadata ID (untuk traceback hasil prediksi nanti)
ID_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_train.parquet'))
ID_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_val.parquet'))
ID_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_test.parquet'))

In [29]:
print(ID_test.head())
print(ID_test.columns)

  kabupaten_kota_asli  tahun
0       Kab. Balangan   2024
1       Kab. Balangan   2025
2     Kota Balikpapan   2024
3     Kota Balikpapan   2025
4         Kab. Banjar   2024
Index(['kabupaten_kota_asli', 'tahun'], dtype='object')


In [30]:
# Load metadata kolom (referensi)   
kolom_info = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'kolom_info.joblib'))

In [31]:
# Verifikasi shape
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}, ID_train: {ID_train.shape}")
print(f"X_val : {X_val.shape}, y_val : {y_val.shape}, ID_val : {ID_val.shape}")
print(f"X_test : {X_test.shape}, y_test : {y_test.shape}, ID_test : {ID_test.shape}")
print(f"\nTotal baris: {len(X_train) + len(X_val) + len(X_test)}")
print(f"Total fitur: {X_train.shape[1]} kolom")
print(f"Target : produktivitas padi (Qu/Ha)")
print(f"Range target train: {y_train.min():.2f} - {y_train.max():.2f}")

X_train: (278, 111), y_train: (278,), ID_train: (278, 2)
X_val : (56, 111), y_val : (56,), ID_val : (56, 2)
X_test : (111, 111), y_test : (111,), ID_test : (111, 2)

Total baris: 445
Total fitur: 111 kolom
Target : produktivitas padi (Qu/Ha)
Range target train: 15.78 - 54.46


In [32]:
# Setup Output Folder
MODEL_FOLDER = os.path.join(BASE_PATH, 'models')
RESULT_FOLDER = os.path.join(BASE_PATH, 'results')

os.makedirs(MODEL_FOLDER, exist_ok=True)
os.makedirs(RESULT_FOLDER, exist_ok=True)

In [33]:
print("Mulai training Linear Regression")

# alpha kecil dulu sebagai baseline regularization
ridge_model = Ridge(alpha=1.0)

# training
ridge_model.fit(X_train, y_train)

print("Training selesai")

Mulai training Linear Regression
Training selesai


In [34]:
# Prediksi
ridge_train_pred = ridge_model.predict(X_train)
ridge_val_pred = ridge_model.predict(X_val)
ridge_test_pred = ridge_model.predict(X_test)

In [35]:
def hitung_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

In [36]:
# evaluasi train
ridge_train_mae, ridge_train_rmse, ridge_train_r2 = hitung_metrics(
    y_train,
    ridge_train_pred
)


In [37]:
# evaluasi validation
ridge_val_mae, ridge_val_rmse, ridge_val_r2 = hitung_metrics(
    y_val,
    ridge_val_pred
)

In [38]:
# evaluasi test
ridge_test_mae, ridge_test_rmse, ridge_test_r2 = hitung_metrics(
    y_test,
    ridge_test_pred
)

In [39]:
print("HASIL RIDGE REGRESSION")

print("\nTrain")
print(f"MAE  : {ridge_train_mae:.4f}")
print(f"RMSE : {ridge_train_rmse:.4f}")
print(f"R2   : {ridge_train_r2:.4f}")

print("\nValidation")
print(f"MAE  : {ridge_val_mae:.4f}")
print(f"RMSE : {ridge_val_rmse:.4f}")
print(f"R2   : {ridge_val_r2:.4f}")

print("\nTest")
print(f"MAE  : {ridge_test_mae:.4f}")
print(f"RMSE : {ridge_test_rmse:.4f}")
print(f"R2   : {ridge_test_r2:.4f}")

HASIL RIDGE REGRESSION

Train
MAE  : 2.2957
RMSE : 3.0493
R2   : 0.7990

Validation
MAE  : 3.1904
RMSE : 4.1626
R2   : 0.5970

Test
MAE  : 3.8970
RMSE : 4.6464
R2   : 0.5971


In [40]:
# Simpan Model Ridge Regression
joblib.dump(
    ridge_model,
    os.path.join(BASE_PATH, 'models', 'ridge_regression.joblib')
)
print("Model berhasil disimpan.")

Model berhasil disimpan.


In [41]:
# Simpan metrics
metrics_ridge = pd.DataFrame({
    "model": ["Ridge Regression"],
    "train_mae": [ridge_train_mae],
    "train_rmse": [ridge_train_rmse],
    "train_r2": [ridge_train_r2],
    "val_mae": [ridge_val_mae],
    "val_rmse": [ridge_val_rmse],
    "val_r2": [ridge_val_r2],
    "test_mae": [ridge_test_mae],
    "test_rmse": [ridge_test_rmse],
    "test_r2": [ridge_test_r2]
})

os.makedirs("results", exist_ok=True)

metrics_ridge.to_csv(
    os.path.join(BASE_PATH, "results", "ridge_regression_metrics.csv"),
    index=False
)
print("Metrics berhasil disimpan.")

Metrics berhasil disimpan.


In [42]:
# Simpan hasil prediksi test
hasil_pred_lr = ID_test.copy()

hasil_pred_lr["actual"] = y_test.values
hasil_pred_lr["prediction_ridge"] = ridge_test_pred

hasil_pred_lr.to_csv(
    os.path.join(BASE_PATH, "results", "ridge_regression_predictions.csv"),
    index=False
)

print("Hasil prediksi berhasil disimpan.")

# preview hasil
print("\nPreview prediction:")
print(hasil_pred_lr.head())

Hasil prediksi berhasil disimpan.

Preview prediction:
  kabupaten_kota_asli  tahun  actual  prediction_ridge
0       Kab. Balangan   2024   36.24         37.010749
1       Kab. Balangan   2025   38.35         36.527533
2     Kota Balikpapan   2024   25.34         32.111926
3     Kota Balikpapan   2025   39.36         32.259769
4         Kab. Banjar   2024   37.79         35.574316
